# 02 — From raw trace to a human-auditable Decision Chain

**Question:** can a reviewer follow the consequential choices without losing the
ability to inspect what actually happened?

This notebook opens one completed chart review at four levels:

1. Codex protocol events — very detailed harness traffic.
2. Canonical Langtrace/Layer-1 events — observable tool calls and server facts.
3. Deterministic ReAct cycles — state before, action, observation, state after.
4. Decision Episodes — one material choice that one human can judge with one verdict.

The Decision Chain is an index over the raw trace, not a replacement for it. Every
episode must retain a path back to its cycle and event evidence. We never display or
claim private chain-of-thought.


## 1. Select a run

The checked demonstration uses the SYNX03 policy-guided Luna run because it contains
the full pattern a reviewer needs to see: inventory, keyword search, candidate-note
selection, three independent evidence judgments, conflict resolution, and a stopping
decision. Override the paths with `ACR_AUDIT_LEDGER` and `ACR_AUDIT_RUN_ID`.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import os

from IPython.display import Markdown, display
from acr.mvp.human_review import human_review_view
from acr.mvp.ledger import SemanticaLedger

START_DIR = Path.cwd().resolve()
ROOT = START_DIR if (START_DIR / "pyproject.toml").is_file() else START_DIR.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"

checked_ledger = ROOT / "runs/notebook-live-20260827/ledger.json"
generated_ledger = ROOT / "runs/postdoc-study/ledger.json"
seed_ledger = ROOT / "runs/policy-experiment-20260827/experiment-ledger.json"
default_ledger = next(
    (path for path in (checked_ledger, generated_ledger, seed_ledger) if path.is_file()),
    generated_ledger,
)
LEDGER_PATH = Path(os.environ.get("ACR_AUDIT_LEDGER", default_ledger))
assert LEDGER_PATH.is_file(), "Run Notebook 1 first or set ACR_AUDIT_LEDGER"
ledger = SemanticaLedger(LEDGER_PATH)

def one_line(value, limit=100):
    text = " ".join(str("" if value is None else value).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

def display_path(value):
    path = Path(value).resolve()
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def markdown_table(rows, columns):
    def safe(value):
        return one_line(value, 120).replace("|", "/")
    return "\n".join([
        "| " + " | ".join(label for _, label in columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
        *("| " + " | ".join(safe(row.get(key, "")) for key, _ in columns) + " |"
          for row in rows),
    ])

preferred_run = (
    "20260827T135252029492Z_SYNX03_STORE_390_date_of_initial_diagnosis_policy_bundle"
)
run_id = os.environ.get("ACR_AUDIT_RUN_ID")
if run_id is None and ledger.selected_analysis(preferred_run):
    run_id = preferred_run
if run_id is None:
    selections = ledger.graph.find_nodes(node_type="AnalysisSelection")
    assert selections, "The ledger has no explicitly selected analysis"
    run_id = str((selections[-1].get("metadata") or {}).get("run_id"))
analysis_id = ledger.selected_analysis(run_id)
assert analysis_id, "Choose/select one reconstruction before human review"
run_dir = LEDGER_PATH.parent / run_id
artifact = ledger.load_analysis_artifact(run_id, analysis_id)
view = human_review_view(ledger, run_id, analysis_id, run_dir=run_dir)
display({
    "run_id": run_id,
    "analysis_id": analysis_id,
    "case": artifact.get("patient_id") or run_id.split("_", 2)[1],
    "task_arm": artifact.get("task_arm"),
    "review_model": artifact.get("review_model"),
    "reconstructor": artifact.get("reconstructor_identity"),
})


## 2. Measure the abstraction ladder

These counts answer “how much does the reviewer have to read by default?” They do not
prove the abstraction is lossless. The fidelity check comes later: each Decision
Episode must preserve drill-down links and every deterministic cycle must be assigned
exactly once as decision-bearing, decision-support, or mechanical.


In [ ]:
protocol_records = [
    json.loads(line) for line in (run_dir / "layer2_codex.jsonl").read_text().splitlines()
    if line.strip()
]
layer1_events = [
    json.loads(line) for line in (run_dir / "trace.jsonl").read_text().splitlines()
    if line.strip()
]
cycles = artifact["cycles"]
episodes = view["episodes"]
steps = view["review_chain"]["steps"]
ladder = [
    {"representation": "Codex protocol", "units": len(protocol_records),
     "default human use": "Harness/debug only"},
    {"representation": "Canonical Langtrace events", "units": len(layer1_events),
     "default human use": "Observable execution evidence"},
    {"representation": "Deterministic ReAct cycles", "units": len(cycles),
     "default human use": "State/action replay"},
    {"representation": "Decision Episodes", "units": len(episodes),
     "default human use": "Primary audit units"},
]
display(Markdown(markdown_table(ladder, [
    ("representation", "Representation"), ("units", "Units"),
    ("default human use", "Role"),
])))


## 3. Read the observable raw trace

The protocol stream contains SDK lifecycle and model transport events. We count its
event types but deliberately do not print model reasoning payloads. The canonical
Layer-1 stream below is the useful raw audit record: tool name, compact action, and
server result shape.


In [ ]:
protocol_types = Counter(str(row.get("type") or row.get("method") or "other")
                         for row in protocol_records)
display(Markdown("**Codex protocol event types (content redacted):** `" +
                 json.dumps(dict(protocol_types.most_common()), ensure_ascii=False) + "`"))

def event_action(event):
    payload = event.get("payload") or {}
    args = payload.get("args") or event.get("args") or {}
    tool = payload.get("tool") or event.get("tool") or ""
    for key in ("decision", "query", "note_id", "standing", "objective", "status"):
        if args.get(key) is not None:
            return f"{key}={args[key]}"
    return ""

raw_rows = []
for event in layer1_events:
    payload = event.get("payload") or {}
    result = payload.get("result") or event.get("result") or {}
    raw_rows.append({
        "seq": event.get("seq"),
        "kind": event.get("kind"),
        "tool": payload.get("tool") or event.get("tool") or "—",
        "action": event_action(event),
        "result": ", ".join(sorted(result)[:6]) if isinstance(result, dict) else "",
    })
display(Markdown(markdown_table(raw_rows, [
    ("seq", "Seq"), ("kind", "Kind"), ("tool", "Tool"),
    ("action", "Compact action"), ("result", "Result fields"),
])))


## 4. Replay the fixed ReAct cycles

A cycle is not automatically a decision. Search calls executing one precommitted
keyword batch can be support; pagination can be mechanical; a `record_finding` that
commits one note's Standing is decision-bearing. Reconstruction may label this fixed
skeleton, but it may not add, remove, reorder, duplicate, or move a cycle.


In [ ]:
annotations = artifact["cycle_annotations"]
if isinstance(annotations, list):
    annotations = {row["cycle_id"]: row for row in annotations}
cycle_rows = []
for index, cycle in enumerate(cycles, 1):
    annotation = annotations[cycle["cycle_id"]]
    observed = (cycle.get("state_after") or {}).get("observed_state") or {}
    tools = [str(action.get("tool") or "") for action in cycle.get("actions") or []]
    cycle_rows.append({
        "n": index,
        "cycle": cycle["cycle_id"].rsplit(":", 1)[-1],
        "role": annotation["role"],
        "function": annotation.get("decision_function") or "—",
        "tools": ", ".join(tools) or "—",
        "receipt": "yes" if cycle.get("has_decision_receipt") else "no",
        "state": (f"surfaced={len(observed.get('surfaced_notes') or [])}; "
                  f"read={len(observed.get('read_notes') or [])}; "
                  f"findings={len((cycle.get('state_after') or {}).get('declared_state', {}).get('findings') or [])}"),
    })
display(Markdown(markdown_table(cycle_rows, [
    ("n", "#"), ("cycle", "Cycle"), ("role", "Role"),
    ("function", "Decision function"), ("tools", "Actions"),
    ("receipt", "Sealed receipt"), ("state", "State after"),
])))


## 5. Follow the human Decision Chain

Read this as if a colleague were explaining the review aloud. At each step ask:

1. Was this the right question at this point?
2. Were the meaningful alternatives represented?
3. Does the evidence/rule actually support the choice?
4. What judgment remained for the model?
5. If this step is wrong, which later steps inherit the problem?

A resolved policy reference proves only that the reference existed. It does not prove
semantic entailment or clinical correctness.


In [ ]:
narrative = ["## The run, one consequential choice at a time"]
for index, step in enumerate(steps, 1):
    grounding = step.get("grounding_assessment") or {}
    flags = ", ".join(item["code"] for item in step.get("review_attention") or []) or "none"
    policy_refs = [
        str(item["rule_id"]) for item in step.get("guidelines") or []
        if item.get("rule_id")
    ]
    narrative.extend([
        f"### {index}. {step['phase_label']} — `{step['decision_function']}/{step['decision_subject']}`",
        f"- **Question before acting:** {step.get('question')}",
        f"- **Choice:** {step.get('decision')}",
        f"- **Stated reason:** {step.get('reason')}",
        f"- **Basis:** {', '.join(step.get('basis_sources') or []) or 'not recorded'}",
        f"- **Applied/offered clause refs shown here:** {', '.join(sorted(set(policy_refs))) or 'none'}",
        f"- **Reference status:** {grounding.get('reference_resolution_status')}",
        f"- **Remaining judgment:** {grounding.get('judgment_mode')}",
        f"- **Review attention:** {flags}",
        f"- **Resulting state:** {step.get('state_result')}",
        "",
    ])
conclusion = view["review_chain"]["conclusion"]
narrative.extend([
    "## Accepted conclusion",
    f"`{json.dumps(conclusion.get('value'), ensure_ascii=False)}`",
    f"\nSubmission explanation: {conclusion.get('reasoning')}",
])
display(Markdown("\n".join(narrative)))


## 6. Drill one flagged Decision back into evidence

We choose the first step with a review-attention flag. The compact chain tells us where
to look; the episode, runtime testimony, raw events, and field provenance tell us what
authority each statement has. This is the safeguard against a fluent reconstruction
hiding an execution error.


In [ ]:
flagged_step = next((step for step in steps if step.get("review_attention")), steps[0])
episode_id = flagged_step["episode_ids"][0]
episode = next(row for row in episodes if row["episode_id"] == episode_id)
artifact_episode = next(
    row for row in artifact["episodes"] if row["episode_id"] == episode_id
)
reconstructed = episode["reconstruction"]
source_event_ids = set(artifact_episode.get("source_event_ids") or [])
source_events = [
    event for event in layer1_events if f"layer1:{event.get('seq')}" in source_event_ids
]
testimony = (episode.get("runtime_testimonies") or [{}])[0]
drill = {
    "question": flagged_step.get("question"),
    "choice": flagged_step.get("decision"),
    "runtime_testimony_ref": testimony.get("testimony_ref"),
    "runtime_because": testimony.get("because"),
    "reconstructed_rationale": reconstructed.get("decision_rationale"),
    "field_provenance": reconstructed.get("field_provenance"),
    "raw_event_ids": sorted(source_event_ids),
    "raw_tools": [((row.get("payload") or {}).get("tool") or row.get("tool"))
                  for row in source_events],
    "review_attention": flagged_step.get("review_attention"),
}
display(drill)
assert episode["bearing_cycle_id"]
assert episode["raw_langtrace_links"]


## 7. Prove the abstraction still indexes the complete cycle skeleton

“Fewer units” is useful only if it does not silently drop a decision-bearing cycle.
The verifier requires every cycle exactly once across Decision Episodes and mechanical
cycles. Separately, every episode must link back to raw trace material.


In [ ]:
episode_cycle_ids = [
    cycle_id for row in artifact["episodes"] for cycle_id in row["source_cycle_ids"]
]
mechanical_cycle_ids = artifact["mechanical_cycle_ids"]
all_cycle_ids = [row["cycle_id"] for row in cycles]
assert len(episode_cycle_ids + mechanical_cycle_ids) == len(all_cycle_ids)
assert set(episode_cycle_ids + mechanical_cycle_ids) == set(all_cycle_ids)
assert len(set(episode_cycle_ids + mechanical_cycle_ids)) == len(all_cycle_ids)
traceable = sum(bool(row["bearing_cycle_id"]) and bool(row["raw_langtrace_links"])
                for row in episodes)
assert traceable == len(episodes)
display({
    "cycles_accounted_for_exactly_once": f"{len(all_cycle_ids)}/{len(all_cycle_ids)}",
    "episodes_with_raw_drilldown": f"{traceable}/{len(episodes)}",
    "mechanical_cycles": len(mechanical_cycle_ids),
    "decision_or_support_cycles": len(episode_cycle_ids),
})


## 8. Compare task-only with policy-bundle behavior on the same case

If the historical experiment ledger is available, this section compares two selected
SYN0001 runs. It does not align steps merely by sequence number; it shows the semantic
function/subject, the chosen outcome, and whether the Decision node had a direct
Semantica `APPLIED_POLICY` binding.

The interesting result is not only that the final dates differ. The detailed run
retrieved and judged a same-day physician note that the task-only run did not use as
establishing evidence. That creates an actionable retrieval/standing/conflict audit
question.


In [ ]:
comparison_path = ROOT / "runs/policy-experiment-20260827/experiment-ledger.json"
comparison_rows = []
if comparison_path.is_file():
    comparison_ledger = SemanticaLedger(comparison_path)
    comparison_runs = [
        "20260827T101823421486Z_SYN0001_STORE_390_date_of_initial_diagnosis_task_only",
        "20260827T105131502195Z_SYN0001_STORE_390_date_of_initial_diagnosis_policy_bundle",
    ]
    edge_rows = [edge.to_dict() if hasattr(edge, "to_dict") else dict(edge)
                 for edge in comparison_ledger.graph.edges]
    for candidate_run in comparison_runs:
        selected = comparison_ledger.selected_analysis(candidate_run)
        candidate_view = human_review_view(
            comparison_ledger,
            candidate_run,
            selected,
            run_dir=comparison_path.parent / candidate_run,
        )
        arm = candidate_view["task_presentation"]["arm_id"]
        for index, step in enumerate(candidate_view["review_chain"]["steps"], 1):
            decision_ids = {
                row["semantica_decision_id"] for row in step["detail_episodes"]
            }
            direct_bindings = sum(
                edge.get("type") == "APPLIED_POLICY"
                and edge.get("source_id") in decision_ids
                for edge in edge_rows
            )
            comparison_rows.append({
                "arm": arm,
                "step": index,
                "point": f"{step['decision_function']}/{step['decision_subject']}",
                "choice": step.get("decision"),
                "policy_bindings": direct_bindings,
            })
    display(Markdown(markdown_table(comparison_rows, [
        ("arm", "Arm"), ("step", "#"), ("point", "Decision point"),
        ("choice", "Choice"), ("policy_bindings", "Direct policy bindings"),
    ])))
else:
    display(Markdown(
        "Historical paired cohort not present. Notebook 1 can generate equivalent "
        "task-only and policy-bundle runs for comparison."
    ))


## 9. What the Decision layer gains and loses

| Gain | Corresponding risk | Safeguard in this notebook |
|---|---|---|
| One verdict per consequential choice | Reconstruction may choose the wrong boundary | Fixed cycles, sealed receipts, two-pass drift, explicit selection |
| Stable function/subject for cross-run comparison | A later taxonomy may reinterpret the run | Taxonomy is post-run and artifacts are append-only |
| Human-readable rationale and state transition | Fluent text may overstate what happened | Field provenance and exact source refs |
| Causal path and review routing | Temporal adjacency may be mistaken for causation | Only explicit evidenced causal assertions enter the audit chain |
| Much shorter default reading path | Incidental low-level errors may be hidden | 100% episode drill-down plus complete cycle accounting |

**Audit rule:** use the Decision Chain to decide *where to inspect*. Use the raw trace
and provenance to decide *what actually happened*. Clinical correctness still requires
a qualified reviewer.


In [ ]:
closure = {
    "schema": "acr.postdoc_audit_walkthrough.v1",
    "run_id": run_id,
    "analysis_id": analysis_id,
    "counts": {
        "protocol_records": len(protocol_records),
        "langtrace_events": len(layer1_events),
        "react_cycles": len(cycles),
        "decision_episodes": len(episodes),
    },
    "all_cycles_accounted_for": True,
    "all_episodes_traceable": traceable == len(episodes),
    "priority_review_count": view["review_chain"]["priority_review_count"],
    "conclusion": view["review_chain"]["conclusion"],
}
output = ROOT / "runs/postdoc-notebook-output/02_audit_walkthrough.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(closure, ensure_ascii=False, indent=2) + "\n")
display(Markdown(
    f"**Notebook 2 closed.** The human path has {len(episodes)} auditable choices; "
    f"all {len(episodes)} retain raw drill-down. Summary: `{display_path(output)}`."
))
